In [1]:
import math_toolkit
math_toolkit.notebook.activate()


# NamedFunction

Define a custom symbolic function once, keep calls readable, and expand the definition only when you ask.

`Function("F")(x, y) @EqDef@ body`  
`Function("B")[n](x) @EqDef@ body`  
`@NamedFunction` for advanced callable definitions


In [2]:
L = Function("L")(p, q) @EqDef@ ((p**2 - q**2) / 2)
lagrangian_identity = L(p, q) @Eq@ L(p, q)

Energy = Function("E")(p, q) @ EqDef(latex=r"\mathcal{E}") @ ((p**2 + q**2) / 2)
energy_identity = Energy(p, q) @Eq@ Energy(p, q)

md("""Pattern-left definitions produce readable equations for each generated function.

- Lagrangian definition: {{ lagrangian_identity }}
- Energy definition: {{ energy_identity }}
""")

Pattern-left definitions produce readable equations for each generated function.

- Lagrangian definition: $L(p, q) = \frac{p^{2}}{2} - \frac{q^{2}}{2}$
- Energy definition: $\mathcal{E}(p, q) = \frac{p^{2}}{2} + \frac{q^{2}}{2}$


## Core behavior

`EqDef` is the default form for giving a name to a formula. Read `Function("F")(x, y) @EqDef@ body` as "define `F(x, y)` by the formula `body`." The left side is the symbolic pattern being defined, and the right side is the formula `body`.

To use the definition later, assign it to a Python name: `F = Function("F")(x) @EqDef@ body`. `EqDef` does not change an existing symbol or function head named `F` behind the scenes.

After the definition, `F(x, y)` stays compact in later formulas. It expands only when you ask for `rewrite("unhold")`, which replaces the named call by the defining expression.

If a call has no registered definition for its number of arguments, it stays opaque under `rewrite("unhold")`. Indexed families also require a matching number of bracket labels; otherwise the call remains an undefined symbolic application.

For a family such as `B[n]`, put the bracket labels directly in the left pattern. With `Function("B")[n](x) @EqDef@ body`, the definition may use `x` for the input and `n` for the family label.

Every symbol or function that appears in the formula must already exist in the current Python scope. The symbols in the left pattern mark placeholders in the already-written formula, but they do not create new names. If you want to write a Python-style definition such as `def F(x): ...`, use the decorated syntax in `Options and Details` instead.

This lets you keep formulas readable while modeling, then expand at the step where algebra, differentiation, substitution, or export needs the full definition.


## Common patterns

Most definitions are compact `EqDef` expressions. Keep the first draft close to the math you would write on paper, then use advanced callable syntax only when the definition needs Python logic.


**One-variable definitions.** Put the single argument in the left call pattern.


In [3]:
Shifted = Function("Shifted")(x) @EqDef@ (x + 1)

Shifted(x) @Eq@ Shifted(x)


Shifted(x) = x + 1

**Multivariate definitions.** Put all formal arguments in the left call pattern.


In [4]:
Bilinear = Function("Bilinear")(x, y) @EqDef@ (x * y + x)

Bilinear(x, y) @Eq@ Bilinear(x, y)


Bilinear(x, y) = x⋅y + x

**Display templates.** When the display is not a simple function-head application, use `latex=` placeholders to place the rendered arguments directly.


In [5]:
SupNorm = Function("SupNorm")(x) @ EqDef(
    latex=r"\left\|#1\right\|_{\infty}",
) @ Abs(x)

SupNorm(x) @Eq@ SupNorm(x)


‖Sup‖(x) = │x│

**Conditional definitions.** Since the right-hand side is an ordinary SymPy expression, `If(...).Then(...).Otherwise(...)` works directly inside `EqDef`.


In [6]:
PositivePart = Function("PositivePart")(x) @EqDef@ If(x > 0).Then(x).Otherwise(0)

PositivePart(x) @Eq@ PositivePart(x)


                  ⎧x  for x > 0
PositivePart(x) = ⎨            
                  ⎩0  otherwise

**Composition before expansion.** Named calls behave like ordinary expressions while they stay opaque.


In [7]:
Potential = Function("Potential")(q) @ EqDef(latex=r"\mathcal{V}") @ (q**2 / 2)

force_balance = diff(Potential(q), q) + F(q)
force_balance


       d               
F(q) + ──(Potential(q))
       dq              

Rewriting exposes the definition inside the derivative, but SymPy keeps the derivative unevaluated until `.doit()` is called.


In [8]:
force_balance @Eq@ force_balance.doit()


       d                          
F(q) + ──(Potential(q)) = q + F(q)
       dq                         

**Indexed families.** Put index placeholders in the left pattern when the definition itself belongs to a labeled family. `Function("Flow")[n](x) @EqDef@ body` lets the right-hand side use `n`, while bracket calls such as `Flow[t](x)` substitute the actual label during expansion.


In [9]:
Flow = Function("Flow")[n](x) @ EqDef(latex=r"\Phi") @ (x + n * f(x))

Flow[t](x)


Flow(x, _NamedFunctionIndexTuple(t))

In [10]:
Flow[t](x) @Eq@ Flow[t](x)


Flow(x, _NamedFunctionIndexTuple(t)) = t⋅f(x) + x

## Options and Details

This section covers advanced definition forms: dispatch, display control, naming options, and Python callable syntax for cases where one compact `EqDef` formula is not enough.

### Arity dispatch and advanced callables

`EqDef` cases dispatch first by the number of ordinary call arguments, then by the number of bracket indexes. A plain `F(x, y) @EqDef@ body` case matches two arguments and no indexes. `F[i, j](x) @EqDef@ body` matches one argument and exactly two bracket indexes.

When no case matches, the call stays opaque under `rewrite("unhold")`. This includes calls with the wrong number of ordinary arguments and indexed calls with the wrong number of bracket labels.

Use `F = F(x, y) @EqDef@ None` to clear the no-index two-argument definition for `F`. Use `F = F[i, j](x) @EqDef@ None` to clear only the one-argument, two-index case. Clearing a specific indexed case can reveal a broader fallback; clearing the fallback leaves unmatched indexed calls opaque.

Decorator syntax is the advanced form for definitions that need Python logic. When the final parameter is named `idx`, it receives the full bracket-index tuple, so the body can branch on special labels, call helper functions, or handle cases that are awkward to express as one symbolic right-hand side.

`EqDef` is the default form because it keeps the definition in ordinary symbolic notation. `@NamedFunction` and `NamedFunction(...)` are advanced callable forms: they receive symbolic arguments as Python parameters and can use Python control flow, helper code, and, when the final parameter is `idx`, the full index tuple for indexed families.

`name=` controls the generated symbolic head, and `latex=` controls display. These options are useful when the Python helper name or left-hand placeholder is not the mathematical name you want readers to see.

**Multiple arities.** Add distinct arities one case at a time by reassigning the generated function class.

In [ ]:
EnergyShape = Function("EnergyShape")
EnergyShape = EnergyShape(p, q) @EqDef@ ((p**2 + q**2) / 2)
EnergyShape = EnergyShape(p, q, t) @EqDef@ ((p**2 + q**2) * (1 + t) / 2)
energy_shape_rows = [
    {"arity": "two arguments", "definition": EnergyShape(p, q) @Eq@ EnergyShape(p, q)},
    {"arity": "three arguments", "definition": EnergyShape(p, q, t) @Eq@ EnergyShape(p, q, t)},
]

md("""Multiple definitions can share a head when their call shapes are distinct.

{table(header=["arity", "definition"])}{{ energy_shape_rows }}
""")

**Conditional dispatch.** This pattern is conditional dispatch: a definition can return `None` for cases it does not want to expand, so those calls stay opaque while matching cases rewrite normally. True multiple dispatch must be defined in one shot; successive `EqDef` definitions with the same number of arguments overwrite each other.


In [12]:
@NamedFunction
def PositiveOnly(x):
    return If(x > 0).Then(x).Otherwise(None)

positive_call = PositiveOnly(2)
negative_call = PositiveOnly(-2)
positive_identity = positive_call @Eq@ positive_call

md("""The procedural definition returns a formula for positive input and leaves unsupported input opaque.

- Positive call: {{ positive_identity }}
- Negative call: {{ negative_call }}
""")

The procedural definition returns a formula for positive input and leaves unsupported input opaque.

- Positive call: $\mathrm{PositiveOnly}(2) = 2$
- Negative call: $\mathrm{PositiveOnly}(-2)$


### LaTeX display templates

`latex=` accepts a rendered head, a full display template, or a callable renderer. If a string contains no template marker, it is treated as the function head: `latex=r"\mathcal{F}"` displays calls as `\mathcal{F}(x, y)`, and indexed calls place bracket labels in the subscript, such as `\mathcal{F}_{n}(x)`.

If the string contains a template marker, it is treated as the complete LaTeX for the applied function. The ordinary argument list is not appended automatically. Template substitutions are display-only; they do not change the symbolic object or the definition used by `rewrite("unhold")`.

Use `#0` for the generated name itself. The name is rendered through the toolkit's ordinary atom-name normalization, so Greek names, multi-letter names, and subscript-style names behave the same way they do for symbols. Use `#1` through `#9` as short aliases for visible call arguments; these are equivalent to `#{arg:1}` through `#{arg:9}`. Use `#*` as a short alias for `#{arg:*}`, which renders all visible call arguments joined by `, `. Use explicit forms such as `#{arg:10}` for multi-digit argument numbers.

For definitions whose final parameter is `idx`, use `#{idx:1}`, `#{idx:2}`, and so on for individual bracket labels, or `#{idx:*}` for all bracket labels joined by `, `. Index placeholders are valid only for index-aware definitions and only when the expression was written with bracket syntax, such as `F[n](x)`.

Every substituted argument or index is rendered through SymPy's active LaTeX printer, so nested expressions, symbols, indexed objects, and other named functions use their normal notebook display. A placeholder that cannot be resolved, such as `#2` on a one-argument call or `#{idx:1}` on a non-indexed function, raises a `ValueError` while rendering.

For more advanced display, pass a callable to `latex=`. The callable receives already-rendered LaTeX strings for the visible call arguments. When the symbolic definition ends with `idx`, the final callable argument is the tuple of rendered index strings. It must return the complete LaTeX string for the applied function.

A bare `#` starts a template slot such as `#1` or `#{idx:1}`. To show a visible hash sign in the rendered math, write `\#`; that passes LaTeX's escaped hash through. Raw `#` characters do not have a normal display use in LaTeX. The macro text `\idx{...}` is not interpreted by `NamedFunction`.


### Alternative syntax

There are three useful definition styles, and they differ mostly in where the placeholders live and how much Python logic the definition can use.

- Pattern-left syntax is the default for symbolic formulas. Write the formal symbols directly on the left, as in `Function("F")(x) @EqDef@ body`, `Function("F")[n](x) @EqDef@ body`, or `A[i] @EqDef@ body` for indexed expression families. The call pattern supplies ordinary arguments, and the bracket pattern supplies indexes, so the right-hand side can use those same placeholders without a separate placeholder list.
- Configured `EqDef(...)` keeps the same pattern-left math syntax but adds options around it. Use `EqDef(name=...)` when the left head is a temporary placeholder, `EqDef(latex=...)` when display should use a different rendered head or template, and `EqDef(indexed=True)` only when you need indexed-family behavior without writing fixed bracket placeholders in the left pattern. The options change naming and display; the left pattern still says which arguments and indexes the formula defines.
- You can also put the argument and index names directly in `EqDef(...)`. A bare function head can use `EqDef(args=x)` or `EqDef(args=x, indices=n)`, and an indexed base can use `EqDef(indices=i)` when the left side has no brackets yet. Do not combine explicit placeholders with the same placeholders on the left: `F(x) @ EqDef(args=x)` duplicates the call arguments, and `F[n] @ EqDef(indices=n)` or `A[i] @ EqDef(indices=i)` duplicates the indexes.
- Decorator syntax moves the placeholders into a Python signature. Use `@NamedFunction` when the definition needs local conditionals, helper calls, conditional dispatch with `None`, or index logic that cannot be described by one fixed left pattern. If the last parameter is `idx`, bracket indexes are passed to that parameter; if not, bracketed calls stay opaque unless another matching indexed definition is added. The tradeoff is that the definition reads as Python code rather than one symbolic infix formula.
- Functional syntax is the same wrapping operation written as a call, such as `F = NamedFunction(helper, name="F")`. Use it when the helper function already exists, when you want to keep the helper under its original Python name, or when choosing the generated symbolic name and display options at the assignment site reads better than a decorator.

For ordinary formulas, prefer pattern-left syntax. Reach for configured `EqDef(...)` when the symbolic formula is still simple but the generated name or display needs customization. Use explicit placeholders when maintaining older code or when the left side is intentionally just a head or base. Reach for decorator syntax when the definition is genuinely procedural or locally dispatched.


**Decorator syntax.** Use the decorator form when a callable definition is clearer than an expression body.


In [13]:
@NamedFunction(name="Well", latex=r"\mathcal{W}")
def quadratic_well(q):
    return q**2 / 2
Well = quadratic_well

Well(q) @Eq@ Well(q)


           2
          q 
Well(q) = ──
          2 

**Functional syntax.** Wrap an existing callable when the helper already exists or should stay separate from the generated head name.


In [14]:
def cubic_shape(x):
    return x**3 - x

K = NamedFunction(cubic_shape, name="K", latex=r"\mathcal{K}")
K(x) @Eq@ K(x)


        3    
K(x) = x  - x

**Indexed dispatch.** Decorator form can branch on the shape or contents of `idx` before returning a symbolic expression.


In [ ]:
@NamedFunction(latex=r"\Psi")
def Stage(x, idx):
    if len(idx) == 1:
        step, = idx
        if step == 0:
            return x
        return x + step * g(x)
    row, col = idx
    return x + row - col

stage_rows = [
    {"index": "0", "definition": Stage[0](x) @Eq@ Stage[0](x)},
    {"index": "t", "definition": Stage[t](x) @Eq@ Stage[t](x)},
    {"index": "i, j", "definition": Stage[i, j](x) @Eq@ Stage[i, j](x)},
]

md("""A procedural indexed function can dispatch on the number and value of labels.

{table(header=["index", "definition"])}{{ stage_rows }}
""")

In [16]:
Ramp = Function("Scratch")(x) @ EqDef(name="Ramp", latex=r"\rho") @ (x + 1)

Ramp(x) @Eq@ Ramp(x)


Ramp(x) = x + 1

In [17]:
Family = Function("Family")[t](x) @EqDef@ (x + 2)

Family[t](x) @Eq@ Family[t](x)


Family(x, _NamedFunctionIndexTuple(t)) = x + 2

## Semantics

`NamedFunction` is symbolic-only. It does not attach numeric implementations or automatic floating-point evaluation hooks. If the definition expands to ordinary SymPy operations, expand first and then pass the expanded expression to the numeric tool you are using.

`EqDef` is pure. It creates and returns a function class; it does not inspect or mutate the containing namespace. Write `F = Function("F")(x) @EqDef@ body` when you want the Python name `F` to refer to the generated class.

Definitions use fixed positional parameters. Keyword-only parameters, `*args`, `**kwargs`, and defaults are intentionally unsupported because each rewrite case must have a stable symbolic arity.

Named-function heads are subscriptable. If a bracketed call has no matching indexed definition, it stays opaque under `rewrite("unhold")`. In decorator form, make the final parameter exactly `idx` when bracket labels should be passed to the definition; otherwise bracketed calls remain opaque. In `EqDef` form, put bracket labels directly in the left pattern when they are part of the formula.

## Pitfalls


**Definitions expand only when requested.** A named call does not expand just because you placed it inside a larger expression. Rewrite at the point where you want the definition to participate.


In [18]:
Lag = Function("Lag")(x) @EqDef@ (x - x[t])

Lag(x) + 1


Lag(x) + 1

Rewrite the expression at the point where you want the defining body to participate.


In [19]:
(Lag(x) + 1) @Eq@ (Lag(x) + 1)


Lag(x) + 1 = x - x[t] + 1

**Expansion does not force later operations to evaluate.** Rewriting exposes the defining body inside whatever expression you already built, but it does not automatically carry out every pending operation created along the way. This is standard SymPy behavior: call `.doit()` when you want unevaluated operations, such as derivatives, to run.


In [20]:
diff(Lag(x), x) @Eq@ diff(Lag(x), x)


d            ∂           
──(Lag(x)) = ──(x - x[t])
dx           ∂x          

Call `.doit()` after rewriting when the result contains unevaluated operations that you want carried out.


In [21]:
diff(Lag(x), x) @Eq@ diff(Lag(x), x).doit()


d             
──(Lag(x)) = 1
dx            

**EqDef placeholders must already exist.** In `EqDef` form, every name in the left pattern and right-hand side must already be defined before Python evaluates the expression. The pattern records placeholders; it does not create Python variables.


In [22]:
try:
    Function("Unknown")(uPrime) @EqDef@ (uPrime + 1)
except NameError as err:
    print(err)


name 'uPrime' is not defined


Create the placeholder as a symbol first, then use it in both the left pattern and the body.


In [23]:
uPrime = Symbol("uPrime")
Known = Function("Known")(uPrime) @EqDef@ (uPrime + 1)

Known(uPrime) @Eq@ Known(uPrime)


Known(u′) = u′ + 1

Alternatively, use decorator syntax when you want standard Python arguments to introduce the formal variables.


In [24]:
@NamedFunction
def KnownDecorated(uPrime):
    return uPrime + 1

KnownDecorated(x) @Eq@ KnownDecorated(x)


KnownDecorated(x) = x + 1

**The EqDef result must be assigned.** A plain function head and a named definition can have the same printed head name, but they are different Python values. If you keep using the old value, you get an undefined-function call rather than the new definition.


In [25]:
F_plain = Function("F")
Function("F")(x) @EqDef@ (x + 1)

F_plain(x)


F(x)

If an expression already contains calls to the old plain head, define the named function and replace that head with the new class. This updates calls such as `F(x**2 + 2)`, not just the literal call `F(x)`.


In [26]:
old_expression = F_plain(x**2 + 2) + 1
F = Function("F")(x) @EqDef@ (x + 1)

updated_expression = old_expression.subs(F_plain, F)
old_expression @Eq@ updated_expression


 ⎛ 2    ⎞        2    
F⎝x  + 2⎠ + 1 = x  + 4

For indexed calls, substitute the old base head. The toolkit keeps the existing bracket indices while replacing the head, so this works for arbitrary indices and arguments.


In [27]:
Indexed_plain = Function("Indexed")
old_indexed_expression = Indexed_plain[i](x**2 + 2) + 1
Indexed = Function("Indexed")[n](x) @EqDef@ (x + n)

updated_indexed_expression = old_indexed_expression.subs(Indexed_plain, Indexed)
old_indexed_expression @Eq@ updated_indexed_expression


                    ⎛ 2    ⎞            2    
Indexed_mt_indexed_2⎝x  + 2⎠ + 1 = i + x  + 3

**Unmatched bracketed calls stay opaque.** Named functions are subscriptable, but bracket labels affect rewrites only when the definition accepts them. Use an indexed left pattern in `EqDef`, or make the final decorator parameter `idx`, when bracket labels are part of the definition.


In [28]:
Energy = Function("Energy")(q,p) @EqDef@ (q**2 + p**2)

Energy[t](q, p)


Energy(q, p, _NamedFunctionIndexTuple(t))

When brackets are part of the definition, put the bracket label in the left pattern and assign the returned function.


In [29]:
EnergyFamily = Function("EnergyFamily")[t](x) @EqDef@ (x + t)

EnergyFamily[t](x) @Eq@ EnergyFamily[t](x)


EnergyFamily(x, _NamedFunctionIndexTuple(t)) = t + x

## See also

[Function](Function.ipynb)  
[Indexed](Indexed.ipynb)  
[Expression](Expression.ipynb)  
[If](If.ipynb)  
[rewrite](rewrite.ipynb)  
[Function families](../tutorials/function_families.ipynb)  
[Composing structured expressions](../tutorials/composing_expressions.ipynb)
